# M11.6 · Fonksiyon yaklaştırma ve derin Q-ağları (DQN)

Tablo sığmayınca (Go'da ~3^361 durum, Atari'de 210×160×3 piksel) değer bir fonksiyonla yaklaşık hesaplanır: $Q_w(s,a)$. Yarı-gradyan güncelleme, hedefin de w'ye bağlı olması sorunu, durum toplama, karo kodlama, "ölümcül üçlü" (yaklaştırma + önyükleme + politika dışı) ve Baird karşı örneği; DQN'nin iki hilesi deneyim tekrarı ve hedef ağ, Huber kaybı, çift DQN hedefi, Atari ön işleme. Saf Python'da 1000 durumlu yürüyüş, NumPy'da Baird ıraksaması ve TDC, PyTorch'ta CartPole'u çözen DQN, sonda binlerce doğrusal politikayı paralel değerlendiren rastgele arama.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m11-6-dqn.html](https://m-gonul.github.io/pytorch-lab/m11-6-dqn.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m11_6_dqn.py
CHECKS = {
 "b": "import random\nimport numpy as np\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef true_values(n=1000, jump=100):\n    s = np.arange(1, n + 1)[:, None]\n    k = np.arange(1, jump + 1)[None, :]\n    P = np.zeros((n, n))\n    r = np.zeros(n)\n    p = 1 / (2 * jump)\n    for sgn in (1, -1):\n        s2 = s + sgn * k\n        inside = (s2 >= 1) & (s2 <= n)\n        rows, cols = np.nonzero(inside)\n        np.add.at(P, (rows, s2[rows, cols] - 1), p)\n        r += p * ((s2 > n).sum(1) - (s2 < 1).sum(1))\n    return np.linalg.solve(np.eye(n) - P, r)\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def ep():\n        e = need(ns, \"walk_episode\")(random.Random(2))\n        ensure(e[0][0] == 500 and e[-1][2] is None and e[-1][1] in (-1.0, 1.0), \"başlangıç ve son\")\n        ensure(all(1 <= abs(c - a) <= 100 for a, _, c in e[:-1]), \"sıçrama 1–100\")\n        ensure(need(ns, \"group\")(1) == 0 and ns[\"group\"](100) == 0 and ns[\"group\"](101) == 1 and ns[\"group\"](1000) == 9, \"gruplar\")\n    R_.test(\"ortam ve gruplar\", ep)\n\n    def fa():\n        V = true_values()\n        rng = random.Random(0)\n        eps = [ns[\"walk_episode\"](rng) for _ in range(5000)]\n        wm = need(ns, \"gradient_mc\")(eps, 2e-4)\n        wt = need(ns, \"semi_gradient_td\")(eps, 2e-3)\n        mu = np.zeros(1000)\n        for e in eps:\n            for s, _, _ in e:\n                mu[s - 1] += 1\n        mu /= mu.sum()\n        g = (np.arange(1000)) // 100\n        vm, vt = np.array(wm)[g], np.array(wt)[g]\n        em, et = np.sqrt(mu @ (vm - V) ** 2), np.sqrt(mu @ (vt - V) ** 2)\n        ensure(em < 0.1 and et < 0.15, f\"değer hatası: MC {em:.3f}, TD {et:.3f}\")\n        ensure(abs(wt[0]) < abs(wm[0]), \"TD uçlarda sıfıra doğru yanlı olmalı\")\n        R_.table(\"Durum toplama (10 grup), 5000 bölüm\", [\"grup\", \"gerçek ort. değer\", \"gradyan MC\", \"yarı-gradyan TD\"],\n                 [[i, f\"{V[100 * i:100 * i + 100].mean():+.3f}\", f\"{wm[i]:+.3f}\", f\"{wt[i]:+.3f}\"] for i in range(10)],\n                 note=f\"μ ağırlıklı √VE: MC {em:.3f}, TD {et:.3f}. MC VE'nin yerel en küçüğüne yakınsar; yarı-gradyan TD, TD sabit noktasına — ve VE ≤ (1/(1−γ))·min VE sınırı γ = 1 için anlamsızdır: uçlarda belirgin yanlılık (Sutton Şekil 9.1–9.2).\")\n    R_.test(\"gradyan MC ve yarı-gradyan TD\", fa)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"yalnız yineleme döngüsü\", lambda: (max_loops(src, 0, func=\"baird_features\"), max_loops(src, 1, func=\"expected_td\"), max_loops(src, 1, func=\"expected_tdc\")))\n    w0 = [1, 1, 1, 1, 1, 1, 10, 1]\n\n    def feats():\n        X = need(ns, \"baird_features\")()\n        close(X @ np.array(w0, float), np.array([3, 3, 3, 3, 3, 3, 12.0]), what=\"v̂ = Xw\")\n        ensure(np.linalg.matrix_rank(X) == 7, \"öznitelikler bağımsız\")\n    R_.test(\"baird_features\", feats)\n\n    def div():\n        mu = np.full(7, 1 / 7)\n        _, h = need(ns, \"expected_td\")(w0, 0.01, 0.99, 1000, mu)\n        w1, _ = ns[\"expected_td\"](w0, 0.01, 0.99, 1, mu)\n        X = ns[\"baird_features\"]()\n        v = X @ np.array(w0, float)\n        close(w1, np.array(w0, float) + 0.01 * (mu * (0.99 * v[6] - v)) @ X, what=\"bir beklenen güncelleme\")\n        _, ho = ns[\"expected_td\"](w0, 0.01, 0.99, 1000, np.eye(7)[6])\n        wt, htc = need(ns, \"expected_tdc\")(w0, 0.005, 0.05, 0.99, 1000, mu)\n        ensure(h[-1] > 10 * h[0], \"düzgün dağılımda TD ıraksamalı\")\n        ensure(ho[-1] <= ho[0], \"politika üstü dağılımda kararlı\")\n        ensure(htc[-1] < htc[0] and np.isfinite(htc).all(), \"TDC kararlı\")\n        vt = X @ wt\n        dl = 0.99 * vt[6] - vt\n        v0 = X @ np.array(w0, float)\n        r = (mu @ dl ** 2) / (mu @ (0.99 * v0[6] - v0) ** 2)\n        pbe = np.abs(X.T @ (mu * dl)).max()\n        ensure(r < 1e-3, f\"TDC: ortalama kare TD hatası 1000 kat düşmeli (oran {r:.1e})\")\n        R_.table(\"‖w‖, γ = 0.99\", [\"yineleme\", \"yarı-gradyan TD (μ düzgün)\", \"yarı-gradyan TD (μ = politika üstü)\", \"TDC (μ düzgün)\"],\n                 [[k, f\"{h[k]:.2f}\", f\"{ho[k]:.2f}\", f\"{htc[k]:.2f}\"] for k in (0, 100, 300, 600, 1000)],\n                 note=f\"TDC: ortalama kare TD hatası {r:.1e} katına iner, ‖Xᵀμδ‖∞ = {pbe:.1e}; ama v̂ ≈ {vt[0]:.2f} ≠ 0: TD hatasını hızla küçültür, gerçek değere (0) ise çok yavaş kayar (Sutton Şekil 11.2 ve 11.6).\")\n    R_.test(\"ıraksama ve TDC\", div)\n",
 "d": "import torch\nimport torch.nn as nn\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def rep():\n        B = need(ns, \"Replay\")(5, 2)\n        B.push(torch.ones(3, 2), torch.tensor([0, 1, 0]), torch.tensor([1.0, 2, 3]), torch.zeros(3, 2), torch.tensor([False, True, False]))\n        ensure(len(B) == 3, \"uzunluk 3\")\n        B.push(torch.full((4, 2), 7.0), torch.tensor([1, 1, 1, 1]), torch.tensor([4.0, 5, 6, 7]), torch.ones(4, 2), torch.tensor([True] * 4))\n        ensure(len(B) == 5, \"kapasitede durmalı\")\n        s, a, r, s2, d = B.sample(1000, torch.Generator().manual_seed(0))\n        ensure(s.shape == (1000, 2) and a.dtype == torch.long and d.dtype.is_floating_point, \"şekil ve tipler\")\n        ensure(set(r.tolist()) == {3.0, 4.0, 5.0, 6.0, 7.0}, \"en eski iki geçiş üzerine yazılmalı\")\n        ensure(bool(((r == 3) == (s[:, 0] == 1)).all()) and bool(((r > 3) == (d == 1)).all()), \"alanlar hizalı\")\n    R_.test(\"Replay\", rep)\n\n    def tgt():\n        qo, qt = nn.Linear(2, 3, bias=False), nn.Linear(2, 3, bias=False)\n        with torch.no_grad():\n            qo.weight.copy_(torch.tensor([[1.0, 0], [0, 1], [0.5, 0.5]]))\n            qt.weight.copy_(torch.tensor([[2.0, 0], [0, 0.5], [1, 1]]))\n        s2 = torch.tensor([[1.0, 3.0], [4.0, 0.0]])\n        r = torch.tensor([1.0, 0.0]); d = torch.tensor([0.0, 1.0])\n        y = need(ns, \"td_target\")(qt, qo, r, s2, d, 0.9, False)\n        close(y.numpy(), torch.tensor([1 + 0.9 * 4.0, 0.0]).numpy(), what=\"DQN hedefi\")\n        y2 = ns[\"td_target\"](qt, qo, r, s2, d, 0.9, True)\n        close(y2.numpy(), torch.tensor([1 + 0.9 * 1.5, 0.0]).numpy(), what=\"çift DQN hedefi (çevrimiçi ağ eylem 1'i seçer, hedef ağ 1.5 der)\")\n        ensure(not y.requires_grad, \"hedef gradyan taşımamalı\")\n    R_.test(\"td_target\", tgt)\n\n    def su():\n        a, b = nn.Linear(2, 2), nn.Linear(2, 2)\n        a0 = [p.detach().clone() for p in a.parameters()]\n        need(ns, \"soft_update\")(a, b, 0.1)\n        for p, p0, pb in zip(a.parameters(), a0, b.parameters()):\n            close(p.detach().numpy(), (0.9 * p0 + 0.1 * pb.detach()).numpy(), what=\"yumuşak güncelleme\")\n    R_.test(\"soft_update\", su)\n\n    def tr():\n        q, lens = need(ns, \"train_dqn\")(30000, 0, True)\n        ev = ns[\"evaluate\"](q)\n        ma = max(sum(lens[i:i + 10]) / 10 for i in range(max(1, len(lens) - 9)))\n        ensure(ma >= 150 and float(ev.mean()) >= 100, f\"CartPole öğrenilmeli (en iyi 10 bölüm ort. {ma:.0f}, değerlendirme {float(ev.mean()):.0f})\")\n        R_.table(\"Çift DQN, CartPole, 30 000 adım\", [\"ölçü\", \"değer\"],\n                 [[\"bölüm sayısı\", len(lens)], [\"ilk 10 bölüm ort. uzunluk\", f\"{sum(lens[:10]) / 10:.0f}\"], [\"en iyi 10 bölüm ort.\", f\"{ma:.0f}\"],\n                  [\"açgözlü değerlendirme (20 bölüm, en fazla 500)\", f\"{float(ev.mean()):.0f}\"]],\n                 note=\"Rastgele politika ≈ 22 adım dayanır. DQN eğrileri gürültülüdür: performans çöküp toparlanabilir — en iyi ağı saklamak pratikte yaygındır.\")\n    R_.test(\"train_dqn\", tr)\n",
 "e": "import math\nimport torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\nG_, MC, MP, L, FM, TAU = 9.8, 1.0, 0.1, 0.5, 10.0, 0.02\n\n\ndef loop_len(w, x, max_len=500):\n    for t in range(max_len):\n        a = 1 if sum(xi * wi for xi, wi in zip(x, w)) > 0 else 0\n        pos, vel, th, om = x\n        f = FM if a else -FM\n        ct, st = math.cos(th), math.sin(th)\n        temp = (f + MP * L * om * om * st) / (MC + MP)\n        thacc = (G_ * st - ct * temp) / (L * (4 / 3 - MP * ct * ct / (MC + MP)))\n        xacc = temp - MP * L * thacc * ct / (MC + MP)\n        x = [pos + TAU * vel, vel + TAU * xacc, th + TAU * om, om + TAU * thacc]\n        if abs(x[0]) > 2.4 or abs(x[2]) > 12 * 2 * math.pi / 360:\n            return t + 1\n    return max_len\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"yalnız adım döngüsü\", lambda: max_loops(src, 1, func=\"rollout_lengths\"))\n    g = torch.Generator().manual_seed(4)\n    W = torch.rand(40, 4, generator=g, dtype=torch.float64) * 2 - 1\n    x0 = (torch.rand(40, 5, 4, generator=g, dtype=torch.float64) - 0.5) * 0.1\n\n    def eq():\n        Lv = need(ns, \"rollout_lengths\")(W, x0)\n        ref = torch.tensor([sum(loop_len(W[p].tolist(), x0[p, e].tolist()) for e in range(5)) / 5 for p in range(40)], dtype=torch.float64)\n        close(Lv.numpy(), ref.numpy(), atol=1e-9, what=\"döngülü değerlendirmeyle aynı uzunluklar\")\n    R_.test(\"rollout_lengths\", eq)\n\n    def rs():\n        Wb, Lb = need(ns, \"random_search\")(4000, 10, 0)\n        ensure(Wb.shape == (4000, 4) and Lb.shape == (4000,), \"şekiller\")\n        ensure(float(Lb.max()) >= 499 and float(Lb.median()) < 100, \"en iyi politika 500'e ulaşmalı; çoğu kötü\")\n        best = Wb[Lb.argmax()]\n        x0b = (torch.rand(1, 100, 4, generator=torch.Generator().manual_seed(9), dtype=torch.float64) - 0.5) * 0.1\n        gen = float(ns[\"rollout_lengths\"](best[None], x0b)[0])\n        t_vec = bench(lambda: ns[\"random_search\"](4000, 10, 0), repeat=1, number=1)\n        t_loop = bench(lambda: [loop_len(Wb[p].tolist(), [0.01, 0, 0.01, 0]) for p in range(200)], repeat=1, number=1) * 20 * 10\n        R_.table(\"4000 doğrusal politika × 10 ortam\", [\"ölçü\", \"değer\"],\n                 [[\"500'e ulaşan politika\", int((Lb >= 499).sum())], [\"ortanca uzunluk\", f\"{float(Lb.median()):.0f}\"],\n                  [\"en iyi politikanın 100 yeni ortamda ort.\", f\"{gen:.0f}\"], [\"süre: tensör\", fmt_time(t_vec)], [\"süre: politika döngüsü (tahmini)\", fmt_time(t_loop)]],\n                 note=\"CartPole doğrusal bir politikayla çözülebilir; 4 parametreli bir 'ağ' için rastgele arama DQN'den çok daha ucuzdur (Mania ve ark. 2018). Hangi yöntemin seçileceği problemin yapısına bağlıdır.\")\n    R_.test(\"random_search\", rs)\n",
 "x1": "from labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        p, v, d = need(ns, \"mc_step\")(-0.5, 0.0, 2)\n        ensure(abs(v - (0.001 - 0.0025 * __import__(\"math\").cos(-1.5))) < 1e-12 and not d, \"fizik\")\n        ensure(ns[\"mc_step\"](-1.19, -0.07, 0)[:2] == (-1.2, 0.0), \"sol duvar\")\n        T = need(ns, \"tiles\")(-0.5, 0.01, 1)\n        ensure(len(T) == 8 and len(set(T)) == 8 and all(i % 3 == 1 for i in T), \"8 farklı karo, eylem kodlu\")\n        res = [need(ns, \"sarsa_mc\")(100, 0.5, s)[1] for s in range(3)]\n        first = sum(r[0] for r in res) / 3\n        last = sum(sum(r[-10:]) / 10 for r in res) / 3\n        ensure(first > 400 and last < 250, f\"bölüm uzunluğu {first:.0f} → {last:.0f}\")\n        R_.table(\"Dağ arabası, yarı-gradyan SARSA, 8 karo katmanı, α = 0.5/8 (3 tohum)\", [\"bölüm\", \"ort. adım\"],\n                 [[k + 1, f\"{sum(r[k] for r in res) / 3:.0f}\"] for k in (0, 1, 4, 9, 24, 49, 99)],\n                 note=\"Araba doğrudan tırmanamaz: önce geri gidip ivme kazanması gerekir. İyimser başlangıç (w = 0) keşfi tek başına sağlar (Sutton Şekil 10.2).\")\n    R_.test(\"dağ arabası\", t)\n",
 "x2": "import numpy as np\nimport cv2\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    rng = np.random.default_rng(1)\n\n    def pp():\n        a = np.zeros((210, 160, 3), np.uint8); b = np.zeros_like(a)\n        a[100:110, 50:60] = (200, 50, 50)\n        b[30:40, 80:90] = (10, 250, 10)\n        out = need(ns, \"preprocess\")(a, b)\n        ensure(out.shape == (84, 84) and out.dtype == np.uint8, \"84×84 uint8\")\n        ensure(out.max() > 100, \"maksimum: iki karedeki nesneler de görünmeli\")\n        x = rng.integers(0, 256, (2, 210, 160, 3), dtype=np.uint8)\n        ref = cv2.resize(cv2.cvtColor(np.maximum(x[0], x[1]), cv2.COLOR_RGB2GRAY), (84, 110), interpolation=cv2.INTER_AREA)[18:102]\n        d = np.abs(ns[\"preprocess\"](x[0], x[1]).astype(int) - ref.astype(int)).max()\n        ensure(d <= 1, f\"OpenCV gri+küçültme ile en fazla 1 fark (bulunan {d})\")\n        R_.table(\"Ön işleme\", [\"aşama\", \"boyut\", \"bayt\"], [[\"ham kare\", \"210×160×3\", 210 * 160 * 3], [\"gri + 110×84\", \"110×84\", 110 * 84], [\"kırpılmış\", \"84×84\", 84 * 84], [\"4 kare yığını\", \"4×84×84\", 4 * 84 * 84]],\n                 note=\"1 M geçişlik tampon float32 ile ~56 GB; uint8 kareler ve s/s′ paylaşımıyla ~7 GB. Maksimum alma, Atari'nin bazı nesneleri yalnız tek karelerde çizmesine karşıdır.\")\n    R_.test(\"preprocess\", pp)\n\n    def fs():\n        S = need(ns, \"FrameStack\")(4)\n        f = [np.full((84, 84), i, np.uint8) for i in range(6)]\n        S.reset(f[0])\n        ensure(S.state().shape == (4, 84, 84) and (S.state() == 0).all(), \"reset k kopya\")\n        for i in range(1, 6):\n            S.push(f[i])\n        ensure([int(S.state()[j, 0, 0]) for j in range(4)] == [2, 3, 4, 5], \"en eski atılır, sıra korunur\")\n        ensure([need(ns, \"clip_reward\")(r) for r in (-7, 0, 0.3, 400)] == [-1, 0, 1, 1], \"ödül kırpma\")\n    R_.test(\"FrameStack ve clip_reward\", fs)\n",
 "x3": "from labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        s = need(ns, \"semi_gradient\")(1.0, 0.1, 0.9, 50)\n        close(s[0], 1 + 0.1 * 0.8, what=\"ilk adım: (1 + α(2γ − 1))w\")\n        close(s[-1], (1 + 0.1 * 0.8) ** 50, rtol=1e-9, what=\"geometrik büyüme\")\n        r = need(ns, \"residual_gradient\")(1.0, 0.1, 0.9, 50)\n        close(r[-1], (1 - 0.1 * 0.64) ** 50, rtol=1e-9, what=\"kalıntı gradyan küçülür\")\n        o = need(ns, \"on_policy\")(1.0, 0.1, 0.9, 50)\n        close(o[0], (1 + 0.08) * (1 - 0.4), what=\"politika üstü bölüm: (1 + α(2γ − 1))(1 − 4α)\")\n        rows = []\n        for g in (0.4, 0.5, 0.6, 0.9, 0.99):\n            rows.append([g, f\"{ns['semi_gradient'](1.0, 0.1, g, 100)[-1]:.3g}\", f\"{ns['residual_gradient'](1.0, 0.1, g, 100)[-1]:.3g}\", f\"{ns['on_policy'](1.0, 0.1, g, 100)[-1]:.3g}\"])\n        R_.table(\"w (başlangıç 1, α = 0.1), 100 güncelleme / bölüm\", [\"γ\", \"yarı-gradyan, politika dışı\", \"kalıntı gradyan\", \"yarı-gradyan, politika üstü\"], rows,\n                 note=\"γ > 0.5'te yalnız s₁→s₂ geçişini güncellemek ıraksar: hedef 2γw, kestirim w'den hızlı büyür. s₂'nin kendi güncellemesi de yapılırsa (politika üstü dağılım) w sıfıra gider. Kalıntı gradyan her zaman kararlıdır ama stokastik ortamda yanlı ve yavaştır.\")\n    R_.test(\"w → 2w\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m11_6_dqn as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Neden yaklaştırma:** Go ~$3^{361}$ durum, Atari durumu 210×160×3 ekran; sürekli durumlar ayrıklaştırılınca patlar. Durumu öznitelik vektörüyle temsil et, $s=(x_1,\dots,x_N)$, ve öğren: doğrusal $Q(s,a)\approx\sum_iw_{a,i}x_i$ ya da sinir ağı $Q(s,a)\approx q(x;w)$.

**Kayıp:** $\tfrac12\big[Q_w(s,a)-(r+\gamma\max_{a'}Q_{w}(s',a'))\big]^2$. Sorun: **w hem kestirimde hem hedefte**; w'yi değiştirince hedef de kayar. Yarı-gradyan yöntemler hedefi sabit sayar (gradyanını almaz); DQN bunu sertleştirir: hedefte ayrı, yavaş güncellenen ağırlıklar $\bar w$.

**Yakınsama:** tablo Q-öğrenme ve doğrusal Q-öğrenme (politika üstü dağılımda) Robbins–Monro koşullarıyla yakınsar. Doğrusal olmayan ağlarla ve politika dışı veriyle garanti yok — **ölümcül üçlü** (fonksiyon yaklaştırma + önyükleme + politika dışı eğitim) ıraksayabilir (Baird 1995).

**DQN = gradyan Q-öğrenme + derin ağ + iki hile:**

1. **Deneyim tekrarı:** $(s,a,r,s')$'leri bir tamponda sakla, her adımda rastgele mini-batch ile öğren. Ardışık örneklerin korelasyonunu kırar, her deneyimi birçok kez kullanır (veri verimliliği), davranış dağılımını eski politikalar üzerinde ortalar (geri besleme döngüsünü önler). Tampondaki veri eski politikalardan geldiği için öğrenme **politika dışı** olmalı.
2. **Hedef ağ:** $\bar w$ her C adımda bir $w$'ye kopyalanır (ya da yumuşak: $\bar w\leftarrow\tau w+(1-\tau)\bar w$).

Ek: Huber kaybı (büyük hatalarda gradyan kırpma), ödül kırpma ($\mathrm{sign}(r)$), kare atlama, ε 1→0.1 doğrusal azalma. **Çift DQN:** seçimi çevrimiçi, değerlendirmeyi hedef ağla yap.

## 1 · Elle

Doğrusal: $\hat q=0.5+1=1.5$, $e=3-1.5=1.5$, $w\leftarrow w+0.1\cdot1.5\cdot x=(0.65,0.8)$.

Huber: $|e|\le1$ için $\tfrac12e^2$, yoksa $|e|-\tfrac12$ → türev $\mathrm{clip}(e,-1,1)$. DQN'nin "hata kırpma"sı budur.

Conv çıktısı: $\lfloor(n-k)/s\rfloor+1$: 84 → 20 → 9 → 7, $7\cdot7\cdot64=3136$ (sonra 512 tam bağlı, sonra eylem başına bir çıktı — tek ileri geçişte tüm Q'lar).


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Doğrusal Q: $x=(1,2)$, $w=(0.5,0.5)$, hedef 3, $\alpha=0.1$**

- `w1`: Güncelleme sonrası $w_1$
- `w2`: Güncelleme sonrası $w_2$

**Kayıp**

- `mse`: $\tfrac12e^2$ kaybının $e=3$'teki türevi
- `hub`: Huber (δ = 1) kaybının $e=3$'teki türevi

**DQN ağı (Nature): girdi 84×84×4**

- `c1`: Conv 8×8, adım 4, 32 kanal → kenar
- `c3`: Ardından 4×4/2 (64) ve 3×3/1 (64): düzleştirilmiş boyut

**Baird: $w=(1,1,1,1,1,1,10,1)$**

- `b1`: $\hat v(s_1)=2w_1+w_8$
- `b7`: $\hat v(s_7)=w_7+2w_8$

In [ ]:
cevaplar = dict(
    w1=None,
    w2=None,
    mse=None,
    hub=None,
    c1=None,
    c3=None,
    b1=None,
    b7=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"w1":[0.65,0],"w2":[0.8,0],"mse":[3,0],"hub":[1,0],"c1":[20,0],"c3":[3136,0],"b1":[3,0],"b7":[12,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

**1000 durumlu rastgele yürüyüş** (Sutton Örnek 9.1), saf Python. Durumlar 1..1000, başlangıç 500; her adımda 1–100 arası düzgün bir sıçrama, sola ya da sağa eşit olasılıkla; 1'in altına çıkış ödül −1, 1000'in üstüne çıkış +1 (bölüm biter). γ = 1.

**Durum toplama:** 10 grup, her biri 100 durum; $\hat v(s)=w_{g(s)}$, gradyan grup göstergesi.

- `walk_episode(rng)` → `[(s, r, s2), …]` (`s2 = None` terminal).
- `group(s, k=10)` → 0..k−1.
- `gradient_mc(episodes, alpha, k=10)` — $w_g\leftarrow w_g+\alpha(G-\hat v(s))$.
- `semi_gradient_td(episodes, alpha, k=10)` — $w_g\leftarrow w_g+\alpha(r+\hat v(s')-\hat v(s))$.

In [ ]:
%%writefile stage_b.py
import random


def walk_episode(rng, n=1000, jump=100):
    ...


def group(s, k=10):
    ...


def gradient_mc(episodes, alpha, k=10):
    ...


def semi_gradient_td(episodes, alpha, k=10):
    ...


rng = random.Random(0)
eps = [walk_episode(rng) for _ in range(200)]
print(semi_gradient_td(eps, 2e-3))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

**Baird karşı örneği** (Sutton Örnek 11.1), NumPy, döngü yalnız yineleme. 7 durum, 8 ağırlık: $\hat v(s_i)=2w_i+w_8$ (i = 1..6), $\hat v(s_7)=w_7+2w_8$. Tüm ödüller 0 (gerçek değerler 0, $w=0$ ile temsil edilebilir). Hedef politika hep "düz" eylemi seçer → $s_7$. Beklenen güncellemeler (dinamik programlama gibi, örnekleme yok), durum ağırlıkları $\mu$:

- `baird_features()` → `(7, 8)`.
- `expected_td(w0, alpha, gamma, n_iter, mu)` → `(w, normlar)`: $w\leftarrow w+\alpha\sum_s\mu_s\,\delta_s\,x(s)$, $\delta_s=\gamma\hat v(s_7)-\hat v(s)$.
- `expected_tdc(w0, alpha, beta, gamma, n_iter, mu)` — gradyan-TD (TDC): ikinci ağırlık $u$ ($u_0=0$):
  $w\leftarrow w+\alpha\sum_s\mu_s[\delta_sx_s-\gamma x_7(x_s^\top u)]$, $u\leftarrow u+\beta\sum_s\mu_s(\delta_s-x_s^\top u)x_s$.

Davranış dağılımı düzgün ($\mu=1/7$) → TD ıraksar; politika üstü ($\mu=e_7$) → kararlı; TDC düzgün dağılımda bile kararlı.

In [ ]:
%%writefile stage_c.py
import numpy as np


def baird_features():
    ...


def expected_td(w0, alpha, gamma, n_iter, mu):
    ...


def expected_tdc(w0, alpha, beta, gamma, n_iter, mu):
    ...


w0 = [1, 1, 1, 1, 1, 1, 10, 1]
print(expected_td(w0, 0.01, 0.99, 500, np.full(7, 1 / 7))[1][-1])

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

**DQN**, CartPole (ortam fiziği verildi: `cartpole_step`, toplu tensörler). Ödül her adımda 1, en fazla 500 adım.

- `Replay(cap, dim)` — dairesel tampon (önceden ayrılmış tensörler): `push(s, a, r, s2, d)` (her biri birinci boyutta k örnek), `sample(b, g)` → `(s, a, r, s2, d)`; `len(buf)`.
- `td_target(q_target, q_online, r, s2, d, gamma, double)` — `no_grad`; DQN: $r+\gamma(1-d)\max Q_{\bar w}(s')$, çift DQN: $r+\gamma(1-d)Q_{\bar w}(s',\arg\max Q_w(s'))$.
- `soft_update(target, online, tau)` — $\bar w\leftarrow\tau w+(1-\tau)\bar w$ (τ = 1: kopyala).
- `train_dqn(steps=30000, seed=0, double=True)` → `(q, bölüm uzunlukları)`: `make_net()`, Adam (1e-3), tampon 50 000, ısınma 1000, batch 64, Huber kaybı, γ = 0.99, ε 1 → 0.05 ilk %30'da doğrusal, hedef ağ her 500 adımda kopya. **Kesme (500. adım) terminal değildir:** `d` yalnız düşüşte 1.

In [ ]:
%%writefile stage_d.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

G_, MC, MP, L, FM, TAU = 9.8, 1.0, 0.1, 0.5, 10.0, 0.02


def cartpole_step(x, a):
    pos, vel, th, om = x.unbind(1)
    f = torch.where(a == 1, FM, -FM).to(x.dtype)
    ct, st = torch.cos(th), torch.sin(th)
    tm, pml = MC + MP, MP * L
    temp = (f + pml * om ** 2 * st) / tm
    thacc = (G_ * st - ct * temp) / (L * (4.0 / 3.0 - MP * ct ** 2 / tm))
    xacc = temp - pml * thacc * ct / tm
    x2 = torch.stack([pos + TAU * vel, vel + TAU * xacc, th + TAU * om, om + TAU * thacc], 1)
    done = (x2[:, 0].abs() > 2.4) | (x2[:, 2].abs() > 12 * 2 * math.pi / 360)
    return x2, done


def make_net():
    return nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))


def evaluate(q, n=20, max_len=500, seed=123):
    g = torch.Generator().manual_seed(seed)
    x = (torch.rand(n, 4, generator=g) - 0.5) * 0.1
    alive = torch.ones(n, dtype=torch.bool)
    lens = torch.zeros(n)
    for _ in range(max_len):
        with torch.no_grad():
            x, d = cartpole_step(x, q(x).argmax(1))
        lens += alive.float()
        alive &= ~d
    return lens


class Replay:
    def __init__(self, cap, dim):
        ...

    def push(self, s, a, r, s2, d):
        ...

    def sample(self, b, g):
        ...

    def __len__(self):
        ...


def td_target(q_target, q_online, r, s2, d, gamma, double):
    ...


def soft_update(target, online, tau):
    ...


def train_dqn(steps=30000, seed=0, double=True):
    ...


q, lens = train_dqn(3000)
print(len(lens), evaluate(q).mean())

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

**Paralel rastgele arama** (Mania ve ark. 2018): $a=\mathbb 1[w^\top x>0]$ doğrusal politikalarını binlercesi birden aynı tensörde değerlendir.

- `rollout_lengths(W, x0, max_len=500)` — `W (P, 4)`, `x0 (P, E, 4)` başlangıç durumları (`float64`); her politika kendi E ortamında; `(P,)` ortalama bölüm uzunluğu. Ölen ortamlar maskelenir; hepsi öldüyse erken çık.
- `random_search(n, n_envs=10, seed=0)` → `(W, L)`: $w\sim U[-1,1]^4$, $x_0\sim U[-0.05,0.05]^4$.

Adım döngüsü serbest; politika/ortam döngüsü yok.

In [ ]:
%%writefile stage_e.py
import math
import torch

G_, MC, MP, L, FM, TAU = 9.8, 1.0, 0.1, 0.5, 10.0, 0.02


def cartpole_step(x, a):
    pos, vel, th, om = x.unbind(-1)
    f = torch.where(a == 1, FM, -FM).to(x.dtype)
    ct, st = torch.cos(th), torch.sin(th)
    tm, pml = MC + MP, MP * L
    temp = (f + pml * om ** 2 * st) / tm
    thacc = (G_ * st - ct * temp) / (L * (4.0 / 3.0 - MP * ct ** 2 / tm))
    xacc = temp - pml * thacc * ct / tm
    x2 = torch.stack([pos + TAU * vel, vel + TAU * xacc, th + TAU * om, om + TAU * thacc], -1)
    done = (x2[..., 0].abs() > 2.4) | (x2[..., 2].abs() > 12 * 2 * math.pi / 360)
    return x2, done


def rollout_lengths(W, x0, max_len=500):
    ...


def random_search(n, n_envs=10, seed=0):
    ...


W, Lr = random_search(500)
print(Lr.max())

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Dağ arabası: karo kodlama ile yarı-gradyan SARSA

**Dağ arabası** (Sutton Örnek 10.1): $v\leftarrow\mathrm{clip}(v+0.001(a-1)-0.0025\cos3p,\pm0.07)$, $p\leftarrow p+v$; $p<-1.2$ ise $p=-1.2, v=0$; $p\ge0.5$ hedef. Her adım −1. Başlangıç $p\sim U[-0.6,-0.4]$, $v=0$.

**Karo kodlama:** 8 karo katmanı, her biri 8×8 (+1 kayma payı) ızgara; katman $t$ konumda $t/8$, hızda $(3t\bmod 8)/8$ hücre kaydırılır. Eylem başına ayrı ağırlıklar. $\hat q(s,a)=\sum_{i\in F(s,a)}w_i$, adım $\alpha/8$.

- `mc_step(p, v, a)`, `tiles(p, v, a)` (8 indis), `sarsa_mc(n_eps, alpha=0.5, seed=0)` → `(w, bölüm uzunlukları)`; ε = 0 — $w=0$ başlangıcı −1 ödüllere göre iyimser olduğu için keşif kendiliğinden olur.

In [ ]:
%%writefile extra_x1.py
import math
import random


def mc_step(p, v, a):
    ...


def tiles(p, v, a, n_tilings=8, n=8):
    ...


def sarsa_mc(n_eps, alpha=0.5, seed=0, n_tilings=8, max_steps=5000):
    ...


w, lens = sarsa_mc(20)
print(lens)

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Atari ön işleme: maksimum, gri ton, küçültme, kırpma, kare yığını

**Atari ön işleme** (Mnih ve ark. 2013/2015), NumPy + OpenCV.

- `preprocess(prev, cur)` — iki RGB kare (210×160×3, uint8): pikselce **maksimum** (titreşen nesneler), gri ton $Y=0.299R+0.587G+0.114B$ (`float`), `cv2.resize(…, (84, 110), interpolation=cv2.INTER_AREA)`, oyun alanını kırp: satır 18..101 → 84×84 `uint8` (yuvarla).
- `FrameStack(k=4)` — `reset(f)` k kopya; `push(f)` en eskiyi atar; `state()` → `(k, 84, 84)`.
- `clip_reward(r)` → `sign(r)`.

In [ ]:
%%writefile extra_x2.py
import numpy as np
import cv2


def preprocess(prev, cur):
    ...


class FrameStack:
    def __init__(self, k=4):
        ...

    def reset(self, f):
        ...

    def push(self, f):
        ...

    def state(self):
        ...


def clip_reward(r):
    ...


rng = np.random.default_rng(0)
a, b = rng.integers(0, 256, (2, 210, 160, 3), dtype=np.uint8)
print(preprocess(a, b).shape)

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Ölümcül üçlünün en küçük örneği: w → 2w

**İki durum:** öznitelikler 1 ve 2, $\hat v=w$ ve $2w$. Geçiş $s_1\to s_2$, ödül 0 (Sutton 11.2).

- `semi_gradient(w, alpha, gamma, n)` — yalnız bu geçiş tekrar tekrar (politika dışı): $w\leftarrow w+\alpha(2\gamma w-w)$.
- `residual_gradient(w, alpha, gamma, n)` — TD hatasının karesinin **gerçek** gradyanı ($\nabla(2\gamma w-w)^2/2$): $w\leftarrow w-\alpha(2\gamma-1)^2w$.
- `on_policy(w, alpha, gamma, n)` — her bölüm $s_1\to s_2$ ve ardından $s_2\to$ terminal (ödül 0); yarı-gradyan iki güncelleme.

Her biri her güncellemeden sonraki w listesini döndürür.

In [ ]:
%%writefile extra_x3.py
def semi_gradient(w, alpha, gamma, n):
    ...


def residual_gradient(w, alpha, gamma, n):
    ...


def on_policy(w, alpha, gamma, n):
    ...


print(semi_gradient(1.0, 0.1, 0.9, 10)[-1])

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Mnih ve ark. (2013): Atari için DQN

[Mnih ve ark. — Playing Atari with Deep Reinforcement Learning (arXiv:1312.5602)](https://arxiv.org/abs/1312.5602)

Makaledeki DQN'yi uygula ve Pong ile Breakout'ta (Gymnasium + ALE) eğit: ön işleme (x2), 4 kare yığını, ağ (16 adet 8×8/4, 32 adet 4×4/2, 256 tam bağlı), RMSProp, mini-batch 32, ε 1'den 0.1'e ilk 1 M karede doğrusal, tampon 1 M, kare atlama k = 4, ödül kırpma. Nature (2015) sürümüne (hedef ağ, 3 conv) ve çift DQN hedefine geç ve farkı ölç.


**Adımlar**

1. Bu modülün d aşamasındaki Replay, td_target ve soft_update'i kullan; tamponu uint8 kareler olarak sakla (bellek!).
2. Ortalama Q değerini sabit bir durum kümesinde izle (makale Şekil 2).
3. Pong'da ≥ +18 ortalama skor; Breakout'ta eğitim eğrisi.

**Kabul ölçütleri**

- Pong: 2–5 M kare içinde öğrenme eğrisi −21'den pozitife.
- Tutulan durumlardaki ortalama max Q düzgün artar (makalenin gözlemi).
- Çift DQN, Q değerlerinin aşırı kestirimini azaltır (van Hasselt 2016 Şekil 3 nitel).

## Makale ödevi: van Hasselt ve ark. (2016): Çift DQN

[van Hasselt, Guez, Silver — Deep Reinforcement Learning with Double Q-learning (AAAI 2016)](https://arxiv.org/abs/1509.06461)

Makalenin Bölüm 3 deneyini yeniden üret: gerçek $Q_*(s,a)=V_*(s)$ (ör. $\sin s$) ve her eylem için ayrı polinom yaklaştırıcı (derece 6), birkaç örnek noktadan uydurulmuş; $\max_aQ_t(s,a)$ ile $V_*$ arasındaki fark ve çift kestirimci. Ardından d aşamasındaki DQN'yi double=False/True ile CartPole'da 5 tohumla çalıştır ve tutulan durumlardaki Q ortalamasını gerçek getiriyle karşılaştır.


**Adımlar**

1. NumPy ile polinom uydurma; eylem sayısı 2, 10 ve 100.

**Kabul ölçütleri**

- Eylem sayısı arttıkça tek kestirimcinin yanlılığı artar, çift kestirimci yanlılığı küçük tutar (makale Şekil 2).

## Kaynaklar

- [Mnih ve ark. — Playing Atari with Deep Reinforcement Learning (arXiv:1312.5602, 2013)](https://arxiv.org/abs/1312.5602) — arXiv
- [Mnih ve ark. — Human-level control through deep reinforcement learning (Nature 2015)](https://doi.org/10.1038/nature14236) — ©
- [van Hasselt, Guez, Silver — Deep Reinforcement Learning with Double Q-learning (AAAI 2016)](https://arxiv.org/abs/1509.06461) — arXiv
- [Sutton & Barto — Reinforcement Learning: An Introduction (2. baskı), Bölüm 9–11](http://incompleteideas.net/book/the-book-2nd.html) — CC BY-NC-ND 2.0
- [Baird — Residual Algorithms: Reinforcement Learning with Function Approximation (ICML 1995)](https://doi.org/10.1016/B978-1-55860-377-6.50013-X) — ©
- [Mania, Guy, Recht — Simple random search provides a competitive approach to RL (NeurIPS 2018)](https://arxiv.org/abs/1803.07055) — arXiv